# Extra B · A IA e a qualidade da água

O satélite não mede a qualidade da água directamente, mas mede a **cor** da água. Com o índice **NDTI**
(verde contra vermelho) vemos quando a água fica mais **barrenta** (turbidez, sedimentos). A plataforma ETE
acompanha 9 troços do rio, todos os meses, desde 2006.

Aqui vamos: 1) ver 20 anos de turbidez em Marracuene; 2) calcular o "normal" de cada mês e as **anomalias**;
3) tentar **prever** os meses com água mais barrenta do que o normal com Machine Learning... e descobrir
porque é que isso falha.


| Sinal | Quer dizer |
| --- | --- |
| ▶️ | Corra a célula: clique nela e prima **Shift + Enter** |
| ✏️ | Edite: troque os `...` pela sua resposta e corra outra vez |
| ✍️ | Responda por escrito, na célula de texto logo abaixo (duplo clique para escrever) |
| ✅ | A célula verifica a sua resposta |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics


def encontrar(nome):                                   # procura o ficheiro na pasta dados/ (ao lado, ou uma pasta acima)
    for pasta in ["dados", os.path.join("..", "dados"), "."]:
        if os.path.exists(os.path.join(pasta, nome)):
            return os.path.join(pasta, nome)
    raise FileNotFoundError(f"Não encontrei {nome}. Abra o Jupyter na pasta Workshop_2/2_pratica.")


serie = pd.read_csv(encontrar("qualidade_agua_mensal.csv"))
print(serie["nome"].unique())

## Passo 1 · 20 anos de turbidez em Marracuene ▶️

`ndti` mais alto = água mais barrenta. Por baixo, o caudal do rio no mesmo mês (dois gráficos separados, com o mesmo
eixo do tempo: dois eixos verticais no mesmo gráfico enganam o olho).

In [ ]:
TROCO = "Q8"                                  # ✏️ experimente "Q1" (fronteira) ou "Q9" (foz)
q = serie[(serie["troco"] == TROCO)].dropna(subset=["ndti"]).copy()
q["data"] = pd.to_datetime(q["month"])

fig, eixos = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
eixos[0].plot(q["data"], q["ndti"], ".-", color="saddlebrown")
eixos[0].set_ylabel("NDTI (turbidez)")
eixos[0].set_title(f"{q['nome'].iloc[0]}: turbidez medida por satélite (em cima) e caudal do rio (em baixo)")
eixos[1].plot(q["data"], q["caudal_m3s"], color="steelblue")
eixos[1].set_ylabel("Caudal (m³/s)")
plt.tight_layout()
plt.show()
print(len(q), "meses com medição, de", q["month"].min(), "a", q["month"].max())

## Passo 2 · O "normal" de cada mês e as anomalias ▶️

A água é naturalmente mais barrenta na época das chuvas. Por isso comparamos cada mês com **o mesmo mês nos
outros anos** (a mediana). A diferença é a **anomalia**: positiva = mais barrenta do que o habitual.

In [ ]:
normal = q.groupby("mes_do_ano")["ndti"].median()
q["anomalia"] = q["ndti"] - q["mes_do_ano"].map(normal)

fig, eixos = plt.subplots(1, 2, figsize=(14, 4))
eixos[0].bar(normal.index, normal.values, color="saddlebrown")
eixos[0].set_title("O normal de cada mês (mediana do NDTI)")
eixos[0].set_xlabel("mês")
eixos[1].bar(q["data"], q["anomalia"], width=25, color=np.where(q["anomalia"] > 0, "saddlebrown", "steelblue"))
eixos[1].set_title("Anomalia: castanho = mais barrenta do que o normal")
plt.tight_layout()
plt.show()
print("Os 5 meses mais barrentos em relação ao normal:")
print(q.nlargest(5, "anomalia")[["month", "ndti", "anomalia", "caudal_m3s", "chuva_mm"]].round(3).to_string(index=False))

## Passo 3 · A IA consegue prever os meses mais barrentos? ▶️

Rótulo: o mês teve água **mais barrenta do que o normal** (classe "elevada" ou "muito elevada" na plataforma).
Características: o caudal, a chuva e o mês. Treinamos com 2006–2018 e testamos com 2019–2026 (**anos que o modelo nunca viu**).

In [ ]:
d = q.dropna(subset=["caudal_m3s", "chuva_mm"]).copy()
d["barrenta"] = d["classe"].isin(["elevada", "muito elevada"]).astype(int)
d["log_caudal"] = np.log1p(d["caudal_m3s"])
CARACTERISTICAS = ["log_caudal", "chuva_mm", "mes_do_ano"]
treino, teste = d[d["ano"] <= 2018], d[d["ano"] >= 2019]

modelo = RandomForestClassifier(n_estimators=300, random_state=0, min_samples_leaf=3).fit(treino[CARACTERISTICAS], treino["barrenta"])
p = modelo.predict_proba(teste[CARACTERISTICAS])[:, 1]
print(f"Meses de treino: {len(treino)} · meses de teste: {len(teste)} ({teste['barrenta'].sum()} barrentos)")
print(f"ROC-AUC no teste: {metrics.roc_auc_score(teste['barrenta'], p):.2f}   (0,5 = ao calhas; 1 = perfeito)")
print(f"Acerto: {metrics.accuracy_score(teste['barrenta'], p > 0.5):.2f}  · acerto de quem diz sempre 'normal': {1 - teste['barrenta'].mean():.2f}")
print("Importâncias:", dict(zip(CARACTERISTICAS, modelo.feature_importances_.round(2))))

✍️ **Pergunta:** o modelo é melhor do que dizer sempre "normal"? O que pode estar a faltar?

*Resposta:*

## O que aprendemos

- **A IA não faz milagres.** Sem as pistas certas, não aprende nada útil, por mais árvores que tenha.
- Em Marracuene, perto da foz, a cor da água depende muito da **maré** (a água do mar entra e sai do rio).
  No projecto ETE, juntar a maré no momento da imagem fez o modelo de turbidez passar de quase inútil para
  uma boa previsão (ROC-AUC perto de 0,9).
- O satélite mede a **cor** da água (turbidez, sedimentos, um sinal de algas e plantas flutuantes, temperatura
  da superfície). **Não** mede pH, oxigénio, nutrientes, metais nem bactérias, e nunca diz se a água é **própria
  para beber**.